# House Prices 데이터 계약 검사

## tl;dr

- `train.csv`는 **1,460행 × 81열**, `test.csv`는 **1,459행 × 80열**이다.
- 테스트 컬럼은 학습 컬럼에서 `SalePrice`만 제외한 결과와 이름·순서가 정확히 같다.
- `sample_submission.csv`는 **1,459행 × 2열**이며, 테스트의 `Id` 1,461–2,919를 같은 순서로 보존한다.
- 제출 계약의 치명적·높음 수준 문제는 발견되지 않았다. 다만 테스트 결측값 때문에 **8개 수치 열**의 pandas 저장 dtype이 학습 데이터와 다르며, `data_description.txt`에는 실제 CSV와 다른 두 개의 오래된 열 이름이 있다.


## Context & Methods

이 노트북은 Kaggle House Prices 원본 파일 네 개가 모델링과 제출에 사용 가능한지 확인하는 데이터 품질 검사다. 한 행은 주택 한 건이며 `Id`를 행 식별자로 본다. 핵심 계약은 train/test 스키마 정합성, 식별자 유일성·순서, 제출 템플릿의 열·행·값 유효성이다.

### Key Assumptions

- 원본 CSV는 수정하지 않는다.
- 문자열 `NA`와 `None`은 구조적 부재일 수 있으므로 원시 토큰 검사에서는 `keep_default_na=False`로 보존한다.
- `sample_submission.csv`의 `SalePrice`는 제출 형식 검사용 벤치마크 값이며 최종 모델 예측이 아니다.
- 분포·성능 분석이 아니라 파일 계약 검사가 범위이므로 차트 대신 작은 검증 표를 사용한다.


## Data

### 1. Load source files

In [1]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"

source_paths = {
    "train": DATA_DIR / "train.csv",
    "test": DATA_DIR / "test.csv",
    "description": DATA_DIR / "data_description.txt",
    "sample_submission": DATA_DIR / "sample_submission.csv",
}

train = pd.read_csv(source_paths["train"])
test = pd.read_csv(source_paths["test"])
sample_submission = pd.read_csv(source_paths["sample_submission"])
train_raw = pd.read_csv(source_paths["train"], keep_default_na=False)
test_raw = pd.read_csv(source_paths["test"], keep_default_na=False)
description_text = source_paths["description"].read_text(errors="replace")


### 2. Record source identity

In [2]:
source_inventory = pd.DataFrame(
    [
        {
            "source": name,
            "file": path.name,
            "bytes": path.stat().st_size,
            "sha256": sha256(path.read_bytes()).hexdigest(),
        }
        for name, path in source_paths.items()
    ]
)
display(source_inventory)


,source,file,bytes,sha256
0,train,train.csv,460676,1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b...
1,test,test.csv,451405,8fdd3d829d4d986b58f845c9553b225e67dd8383624d90...
2,description,data_description.txt,13370,356a4670aa688afa8ba44b1365b633fbbac2dae535fe46...
3,sample_submission,sample_submission.csv,31939,f19e99699b925aa3085c8d2ae43c62f15fbcd35e4a953b...


## Results

### 3. Validate file shapes and column contracts

In [3]:
expected_test_columns = train.drop(columns="SalePrice").columns.tolist()
contract_checks = pd.DataFrame(
    [
        {
            "check": "test columns equal train minus target",
            "passed": test.columns.tolist() == expected_test_columns,
            "evidence": f"train={train.shape}, test={test.shape}",
        },
        {
            "check": "sample columns and order",
            "passed": sample_submission.columns.tolist() == ["Id", "SalePrice"],
            "evidence": str(sample_submission.columns.tolist()),
        },
        {
            "check": "sample row count equals test",
            "passed": len(sample_submission) == len(test),
            "evidence": f"sample={len(sample_submission)}, test={len(test)}",
        },
        {
            "check": "sample Id equals test Id in order",
            "passed": sample_submission["Id"].equals(test["Id"]),
            "evidence": f"first={test['Id'].iloc[0]}, last={test['Id'].iloc[-1]}",
        },
    ]
)
display(contract_checks)
assert contract_checks["passed"].all(), "Critical file-contract check failed"


,check,passed,evidence
0,test columns equal train minus target,True,"train=(1460, 81), test=(1459, 80)"
1,sample columns and order,True,"['Id', 'SalePrice']"
2,sample row count equals test,True,"sample=1459, test=1459"
3,sample Id equals test Id in order,True,"first=1461, last=2919"


### 4. Validate identifiers, required values, and duplicates

In [4]:
id_checks = pd.DataFrame(
    [
        {
            "dataset": "train",
            "rows": len(train),
            "id_min": train["Id"].min(),
            "id_max": train["Id"].max(),
            "id_nulls": train["Id"].isna().sum(),
            "duplicate_ids": train["Id"].duplicated().sum(),
            "exact_duplicate_rows": train.duplicated().sum(),
        },
        {
            "dataset": "test",
            "rows": len(test),
            "id_min": test["Id"].min(),
            "id_max": test["Id"].max(),
            "id_nulls": test["Id"].isna().sum(),
            "duplicate_ids": test["Id"].duplicated().sum(),
            "exact_duplicate_rows": test.duplicated().sum(),
        },
        {
            "dataset": "sample_submission",
            "rows": len(sample_submission),
            "id_min": sample_submission["Id"].min(),
            "id_max": sample_submission["Id"].max(),
            "id_nulls": sample_submission["Id"].isna().sum(),
            "duplicate_ids": sample_submission["Id"].duplicated().sum(),
            "exact_duplicate_rows": sample_submission.duplicated().sum(),
        },
    ]
)
display(id_checks)

assert train["SalePrice"].notna().all()
assert set(train["Id"]).isdisjoint(set(test["Id"]))
assert id_checks[["id_nulls", "duplicate_ids", "exact_duplicate_rows"]].to_numpy().sum() == 0


,dataset,rows,id_min,id_max,id_nulls,duplicate_ids,exact_duplicate_rows
0,train,1460,1,1460,0,0,0
1,test,1459,1461,2919,0,0,0
2,sample_submission,1459,1461,2919,0,0,0


### 5. Validate benchmark submission values

In [5]:
submission_value_checks = pd.DataFrame(
    [
        {
            "dtype": str(sample_submission["SalePrice"].dtype),
            "nulls": sample_submission["SalePrice"].isna().sum(),
            "finite": bool(np.isfinite(sample_submission["SalePrice"]).all()),
            "all_positive": bool((sample_submission["SalePrice"] > 0).all()),
            "minimum": sample_submission["SalePrice"].min(),
            "maximum": sample_submission["SalePrice"].max(),
            "distinct_values": sample_submission["SalePrice"].nunique(),
        }
    ]
)
display(submission_value_checks)
assert submission_value_checks.loc[0, ["nulls"]].item() == 0
assert submission_value_checks.loc[0, "finite"]
assert submission_value_checks.loc[0, "all_positive"]


,dtype,nulls,finite,all_positive,minimum,maximum,distinct_values
0,float64,0,True,True,135751.318893,281643.976117,1438


### 6. Explain train/test dtype differences

In [6]:
dtype_mismatches = pd.DataFrame(
    [
        {
            "column": column,
            "train_dtype": str(train[column].dtype),
            "test_dtype": str(test[column].dtype),
            "train_nulls": int(train[column].isna().sum()),
            "test_nulls": int(test[column].isna().sum()),
        }
        for column in test.columns
        if train[column].dtype != test[column].dtype
    ]
)
display(dtype_mismatches)


,column,train_dtype,test_dtype,train_nulls,test_nulls
0,BsmtFinSF1,int64,float64,0,1
1,BsmtFinSF2,int64,float64,0,1
2,BsmtUnfSF,int64,float64,0,1
3,TotalBsmtSF,int64,float64,0,1
4,BsmtFullBath,int64,float64,0,2
5,BsmtHalfBath,int64,float64,0,2
6,GarageCars,int64,float64,0,1
7,GarageArea,int64,float64,0,1


### 7. Preserve sentinel meaning and audit column documentation

In [7]:
sentinel_counts = pd.DataFrame(
    [
        {
            "dataset": name,
            "literal_NA": int((frame == "NA").to_numpy().sum()),
            "literal_None": int((frame == "None").to_numpy().sum()),
            "empty_strings": int((frame == "").to_numpy().sum()),
        }
        for name, frame in [("train", train_raw), ("test", test_raw)]
    ]
)
display(sentinel_counts)

described_columns = [
    match.group(1).strip()
    for match in re.finditer(r"^([^\s][^:\n]*):", description_text, re.MULTILINE)
]
predictor_columns = [
    column for column in train.columns if column not in {"Id", "SalePrice"}
]
documentation_audit = pd.DataFrame(
    [
        {
            "measure": "predictor columns in CSV",
            "value": len(predictor_columns),
        },
        {
            "measure": "headings in data_description.txt",
            "value": len(described_columns),
        },
        {
            "measure": "CSV names missing from description",
            "value": sorted(set(predictor_columns) - set(described_columns)),
        },
        {
            "measure": "description names absent from CSV",
            "value": sorted(set(described_columns) - set(predictor_columns)),
        },
    ]
)
display(documentation_audit)


,dataset,literal_NA,literal_None,empty_strings
0,train,6965,864,0
1,test,7000,878,0


,measure,value
0,predictor columns in CSV,79
1,headings in data_description.txt,79
2,CSV names missing from description,"[BedroomAbvGr, KitchenAbvGr]"
3,description names absent from CSV,"[Bedroom, Kitchen]"


### 8. Save an inspectable metrics artifact

In [8]:
metrics = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "sources": source_inventory.to_dict(orient="records"),
    "shapes": {
        "train": list(train.shape),
        "test": list(test.shape),
        "sample_submission": list(sample_submission.shape),
    },
    "contract_checks": {
        row["check"]: bool(row["passed"])
        for row in contract_checks.to_dict(orient="records")
    },
    "ids": {
        "train_min": int(train["Id"].min()),
        "train_max": int(train["Id"].max()),
        "test_min": int(test["Id"].min()),
        "test_max": int(test["Id"].max()),
        "train_test_overlap": int(len(set(train["Id"]) & set(test["Id"]))),
        "sample_matches_test_order": bool(sample_submission["Id"].equals(test["Id"])),
    },
    "duplicates": {
        "train_exact": int(train.duplicated().sum()),
        "test_exact": int(test.duplicated().sum()),
        "sample_exact": int(sample_submission.duplicated().sum()),
    },
    "sample_submission": {
        "id_dtype": str(sample_submission["Id"].dtype),
        "saleprice_dtype": str(sample_submission["SalePrice"].dtype),
        "saleprice_nulls": int(sample_submission["SalePrice"].isna().sum()),
        "saleprice_finite": bool(np.isfinite(sample_submission["SalePrice"]).all()),
        "saleprice_positive": bool((sample_submission["SalePrice"] > 0).all()),
        "saleprice_min": float(sample_submission["SalePrice"].min()),
        "saleprice_max": float(sample_submission["SalePrice"].max()),
    },
    "dtype_mismatches": dtype_mismatches.to_dict(orient="records"),
    "sentinel_counts": sentinel_counts.to_dict(orient="records"),
    "documentation": {
        "predictor_count": len(predictor_columns),
        "description_heading_count": len(described_columns),
        "missing_csv_names": sorted(set(predictor_columns) - set(described_columns)),
        "extra_description_names": sorted(set(described_columns) - set(predictor_columns)),
    },
}

metrics_path = REPORTS_DIR / "data_contract_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False) + "\n")
metrics_path


PosixPath('/Users/joonha/workspace/House_Prices/reports/data_contract_metrics.json')

## Takeaways

1. 네 파일은 House Prices 모델링과 제출을 시작하기에 적합하다. train/test 스키마, 식별자, 제출 템플릿의 핵심 계약 검사를 모두 통과했다.
2. `sample_submission.csv`의 `SalePrice`는 유한한 양수 벤치마크 값이지만 최종 예측은 아니다. 실제 제출 때 같은 `Id`와 순서를 유지하고 이 열만 모델 예측으로 교체해야 한다.
3. 테스트 데이터에서 결측값이 있는 8개 수치 열은 `float64`로 읽히므로, dtype을 억지로 정수로 맞추기보다 fold 내부 수치형 결측 대치를 적용해야 한다.
4. `NA`와 `None`은 구조적 부재를 뜻할 수 있어 기본 null 파싱 결과를 그대로 의미적 결측으로 해석하면 안 된다.
5. `data_description.txt`의 `Bedroom`, `Kitchen`은 각각 CSV의 `BedroomAbvGr`, `KitchenAbvGr`로 매핑해야 한다. 이는 낮은 심각도의 문서 이름 불일치다.
